# Task 2: Model Building and Training

**Objective**: Train and compare classification models for fraud detection

**Models to Train**:
1. Logistic Regression (baseline, interpretable)
2. Random Forest (ensemble, good performance)
3. XGBoost (ensemble, best performance)

**Evaluation Metrics**: Accuracy, Precision, Recall, F1-Score, ROC-AUC, AUC-PR

**Strategy**: Stratified K-Fold Cross-Validation (k=5)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import DataLoader
from src.geolocation import IPGeolocation
from src.feature_engineering import FraudFeatureEngineer, CreditCardFeatureEngineer
from src.preprocessing import FraudPreprocessor, ImbalancedDataHandler
from src.model_training import FraudDetectionModels
from src.model_evaluation import ModelEvaluator

print('All modules imported successfully')

## 1. Load and Prepare Data

In [ ]:
# Load datasets
loader = DataLoader('../data')
fraud_data = loader.load_fraud_data()
ip_mapping = loader.load_ip_mapping()

print(f'Fraud Data shape: {fraud_data.shape}')
print(f'Class distribution:\n{fraud_data["class"].value_counts()}')

## 2. Add Geolocation

In [ ]:
# Add country information
geo = IPGeolocation(ip_mapping)
fraud_data = geo.enrich_with_country(fraud_data)

print(f'Unique countries: {fraud_data["ip_country"].nunique()}')
print(f'Sample countries:\n{fraud_data["ip_country"].value_counts().head()}')

## 3. Feature Engineering

In [ ]:
# Engineer features
engineer = FraudFeatureEngineer()
fraud_data = engineer.engineer_time_features(fraud_data)
fraud_data = engineer.engineer_time_since_signup(fraud_data)
fraud_data = engineer.engineer_device_features(fraud_data)
fraud_data = engineer.engineer_categorical_features(fraud_data)
fraud_data = engineer.engineer_user_behavior_features(fraud_data)

print(f'Engineered features: {fraud_data.shape[1]}')
print(f'Sample features:\n{fraud_data.dtypes}')

## 4. Preprocessing

In [ ]:
# Prepare for modeling
preprocessor = FraudPreprocessor()

# Define feature columns
categorical_cols = ['source', 'browser', 'sex', 'ip_country']
numerical_cols = [col for col in fraud_data.columns 
                  if fraud_data[col].dtype in ['int64', 'float64'] 
                  and col not in ['user_id', 'device_id', 'class']]

# Preprocess
fraud_data_clean = preprocessor.prepare_for_modeling(
    fraud_data, 
    target_col='class',
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    fit=True
)

print(f'Cleaned data shape: {fraud_data_clean.shape}')

## 5. Train-Test Split

In [ ]:
# Separate features and target
X = fraud_data_clean.drop('class', axis=1)
y = fraud_data_clean['class']

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'\nTraining set fraud rate: {y_train.mean():.2%}')
print(f'Test set fraud rate: {y_test.mean():.2%}')

## 6. Handle Class Imbalance

In [ ]:
# Apply SMOTE on training set
imbalance_handler = ImbalancedDataHandler(smote_ratio=0.5, random_state=42)
X_train_resampled, y_train_resampled = imbalance_handler.apply_smote(X_train, y_train)

print(f'Original training set - Fraud rate: {y_train.mean():.2%}')
print(f'Resampled training set - Fraud rate: {y_train_resampled.mean():.2%}')

## 7. Train Models

In [ ]:
# Initialize model trainer
trainer = FraudDetectionModels(random_state=42)

# Train and evaluate models
results_df, predictions = trainer.train_and_evaluate_models(
    X_train_resampled, y_train_resampled, X_test, y_test
)

print(results_df.to_string(index=False))

## 8. Model Comparison

In [ ]:
# Plot model comparison
import matplotlib.pyplot as plt

fig = ModelEvaluator.plot_model_comparison(results_df, metric='f1')
plt.show()

# Best model
best_model_idx = results_df['F1-Score'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Model']
print(f'\nBest Model: {best_model_name}')
print(f'F1-Score: {results_df.loc[best_model_idx, "F1-Score"]:.4f}')

## 9. Cross-Validation

In [ ]:
# Perform 5-fold stratified cross-validation
best_model = trainer.models['xgboost']

cv_results = trainer.cross_validate_model(
    best_model, X_train_resampled, y_train_resampled, cv=5
)

# Summarize results
print('Cross-Validation Results:')
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    scores = cv_results[f'test_{metric}']
    print(f'{metric.upper()}: {scores.mean():.4f} (+/- {scores.std():.4f})')

## 10. Save Models

In [ ]:
# Save trained models
trainer.save_model('logistic_regression', '../models/lr_model.pkl')
trainer.save_model('random_forest', '../models/rf_model.pkl')
trainer.save_model('xgboost', '../models/xgb_model.pkl')

print('Models saved successfully')